## Demo 2: StackExchange

The data of a StackExchange site, published as one XML file per table on archive.org.

This demo is about getting files into databases, and about moving data from one database into another one. We start by looking at the data, without touching a database at all.

### What data do we have?

In [ ]:
import os

data_path = r"..\data\stackexchange"

for file in sorted(os.listdir(data_path)):
    size = os.path.getsize(os.path.join(data_path, file))
    print(f"{file:20} {size / 1024 / 1024:6.1f} MB")

### The files are XML, but they are also line oriented

That is what makes them pleasant to work with: the whole file is valid XML, and every single row is valid XML on its own. So we never have to load the whole document into memory.

In [ ]:
users_path = r"..\data\stackexchange\Users.xml"

with open(users_path, encoding="utf-8-sig") as file:
    users_lines = file.readlines()

print(len(users_lines), "lines")
print(users_lines[0], end="")
print(users_lines[1], end="")
print(users_lines[2][:110], "...")
print(users_lines[-1])

#### Why `utf-8-sig` and not `utf-8`?

The file starts with a byte order mark. `Get-Content` in PowerShell removes it without telling us, `open` in Python does not. With plain `utf-8` the first line starts with an invisible `\ufeff`, and a test like `line.startswith("<?xml")` is suddenly false.

In [ ]:
for encoding in ["utf-8", "utf-8-sig"]:
    with open(users_path, encoding=encoding) as file:
        first_line = file.readline()

    print(f"{encoding:10} {first_line[:24]!r:36} starts with '<?xml': {first_line.startswith('<?xml')}")

### One line is one row

In PowerShell we cast the line to `[xml]` and get an `XmlElement` with one property per attribute. In Python we parse the line and take its attributes, and what we get is a plain dictionary.

In [ ]:
import xml.etree.ElementTree as ET

line = users_lines[2]

row = ET.fromstring(line).attrib

row

In [ ]:
print(type(row))
print(type(row["Id"]), repr(row["Id"]))

Every value is a string, in both languages. Nobody has converted anything yet - the types will come from the target table.

### Not every row has every attribute

And this is where the two languages really differ. An attribute that is not in the line is simply not in the dictionary.

In [ ]:
from collections import Counter

attribute_counts = Counter()
row_count = 0

for line in users_lines:
    if line.lstrip().startswith("<row"):
        row_count += 1
        attribute_counts.update(ET.fromstring(line).attrib.keys())

for attribute, count in attribute_counts.most_common():
    print(f"{attribute:16} {count:6} of {row_count}")

In [ ]:
# Find the first user without a Location

for line in users_lines:
    if line.lstrip().startswith("<row"):
        sparse_row = ET.fromstring(line).attrib
        if "Location" not in sparse_row:
            break

print(sparse_row["DisplayName"], "has no Location")

# PowerShell gives us $null for a missing property, and so does .get()
print(sparse_row.get("Location"))

In [ ]:
# But asking for it directly is an error, not a None

sparse_row["Location"]

So a Python port of the import cannot simply read `row[column]` for every column of the target table. It either asks with `.get()`, or it has to know which attributes are there.

### From lines to a data frame

pandas has its own answer to the missing attributes: it collects every key it sees and fills the gaps with `NaN`.

In [ ]:
import pandas as pd

users = pd.DataFrame(
    ET.fromstring(line).attrib
    for line in users_lines
    if line.lstrip().startswith("<row")
)

users

In [ ]:
users.info()

Twelve columns, all of them strings, and three of them with missing values. Converting those strings into the types of a database table is the next step.

### Setting up the connection to SQL Server

Same three lines as in the first demo, only the database and the login are different.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../lib").resolve()))

# The counterpart of Import-Module PSFramework in the sibling: the bulk-load progress
# on screen, everything in demo.log, and every message kept so it can be queried later.
from configure_logging import configure_logging

messages = configure_logging()

from connect_sql_instance import connect_sql_instance
from import_sql_table import import_sql_table
from invoke_sql_query import invoke_sql_query

connection = connect_sql_instance(
    instance="127.0.0.1",
    database="StackExchange",
    username="StackExchange",
    password="Passw0rd!"
)

### What does the target table look like?

The file gives us strings. The table decides what they have to become. `cursor.description` tells us the columns and, for each one, the Python type that pyodbc expects. It is what `GetSchemaTable()` is for the PowerShell version.

In [ ]:
cursor = connection.cursor()
cursor.execute("SELECT TOP 0 * FROM dbo.Users")
description = cursor.description
cursor.close()

for name, type_code, _, _, _, _, null_ok in description:
    print(f"{name:16} {type_code.__name__:10} null_ok={null_ok}")

#### The file and the table do not agree

The table has fourteen columns, a row in the file has twelve attributes, and not even the same twelve on every row.

In [ ]:
table_columns = [column[0] for column in description]

print("in the table but never in the file:", [c for c in table_columns if c not in attribute_counts])
print("in the file but not in the table:  ", [a for a in attribute_counts if a not in table_columns])

### Importing the file

`import_sql_table` reads the file line by line, so the size of the file does not matter. For every line it builds one value per column of the *target* table: it asks the row with `.get()`, so a missing attribute becomes `NULL`, converts the string with the type from `cursor.description`, and sends the rows to the database in batches.

In [ ]:
import_sql_table(
    connection=connection,
    path=r"..\data\stackexchange\Users.xml",
    table="dbo.Users",
    batch_size=5000,
    truncate_table=True
)

In [ ]:
invoke_sql_query(
    connection=connection,
    query='SELECT TOP 5 Id, DisplayName, Location, CreationDate, Reputation FROM dbo.Users'
)

The dates are dates and the numbers are numbers, and the two columns that the file never mentions are `NULL` for every row - just like the `Location` of the users that did not fill it in.

In [ ]:
invoke_sql_query(connection=connection, query="""
SELECT COUNT(*) AS ImportedRows,
       SUM(CASE WHEN Location IS NULL THEN 1 ELSE 0 END) AS NullLocation,
       SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS NullAge
FROM dbo.Users""")

### When the file names a column differently

Badges are created on `Date`, but every table in this database calls that column `CreationDate`.

In [ ]:
badges_path = r"..\data\stackexchange\Badges.xml"

with open(badges_path, encoding="utf-8-sig") as file:
    badges_lines = file.readlines()

ET.fromstring(badges_lines[2]).attrib

In [ ]:
import_sql_table(
    connection=connection,
    path=badges_path,
    table="dbo.Badges",
    batch_size=5000,
    truncate_table=True,
    column_map={"CreationDate": "Date"}
)

In [ ]:
invoke_sql_query(connection=connection, query='SELECT TOP 5 * FROM dbo.Badges')

### The one thing that is really different

The PowerShell version fills a `DataTable` whose columns are typed from `GetSchemaTable()`, and lets ADO.NET convert the strings on the way in. pyodbc has nothing like that: in fast bulk mode it binds a value by its Python type, so a string never reaches an `INT` column.

So `import_sql_table` carries a small table of converters - `int`, `str`, `datetime.fromisoformat` - and picks one per column from `cursor.description`. That table is the part of this function with no counterpart in the sibling repository.

### The same file, into PostgreSQL

Now the second database. The functions are deliberately named and shaped like the SQL Server ones, so a call site looks almost the same. Two things underneath are genuinely different.

In [ ]:
from connect_pg_instance import connect_pg_instance
from import_pg_table import import_pg_table
from invoke_pg_query import invoke_pg_query

pg_connection = connect_pg_instance(
    instance="127.0.0.1",
    database="stackexchange",
    username="stackexchange",
    password="Passw0rd!"
)

#### First difference: PostgreSQL folds identifiers to lower case

The tables are created as `CREATE TABLE Users (Id INT ...)` without quotes, so the catalog holds `users` and `id`. The attributes in the file are `Id`, `AboutMe`, `CreationDate`.

In [ ]:
cursor = pg_connection.cursor()
cursor.execute("SELECT * FROM Users WHERE 1=0")
pg_columns = [column.name for column in cursor.description]
cursor.close()

print("columns in PostgreSQL:", pg_columns)
print()
print("exact matches with the XML attributes:", [c for c in pg_columns if c in attribute_counts])
print("case insensitive matches:             ",
      [c for c in pg_columns if c.lower() in {a.lower() for a in attribute_counts}])

Not a single column matches by name.

In PowerShell this never comes up: `$row.aboutme` finds the `AboutMe` attribute, because property access is case insensitive. A Python dictionary is not. So both import functions lower case the keys of a row and the names of the columns before matching them.

Without that, the import would write 12220 rows of `NULL` and report success - no error anywhere.

In [ ]:
import_pg_table(
    connection=pg_connection,
    path=r"..\data\stackexchange\Users.xml",
    table="Users",
    batch_size=5000,
    truncate_table=True
)

In [ ]:
invoke_pg_query(
    connection=pg_connection,
    query="SELECT id, displayname, location, creationdate, reputation FROM Users LIMIT 5"
)

#### Second difference: COPY instead of INSERT

`import_sql_table` builds an `INSERT` and sends batches through `executemany`, and it has to convert every string into the type of its column first, because pyodbc binds a value by its Python type.

PostgreSQL has `COPY`, which takes the text and parses it into the column type itself. So `import_pg_table` carries no converters at all. Measured on this file, 12220 rows:

| | |
| --- | --- |
| `executemany`, converted values | 1.27 s |
| `executemany`, raw strings | 0.95 s |
| `COPY`, converted values | 0.30 s |
| `COPY`, raw strings | 0.14 s |

The fastest way is also the one with the least code - the opposite of SQL Server, where passing raw strings fails outright.

In [ ]:
import_pg_table(
    connection=pg_connection,
    path=badges_path,
    table="Badges",
    batch_size=10000,
    truncate_table=True,
    column_map={"CreationDate": "Date"}
)

In [ ]:
invoke_pg_query(connection=pg_connection, query="SELECT * FROM Badges LIMIT 5")

The same call with the same parameters against a second database system - and underneath, one function converts every value by hand while the other hands the text straight to the database.

### And the same file into a third database: Oracle

SQL Server had to convert every value. PostgreSQL had to convert none. Oracle is the third answer to the same question, and it is the one worth spending time on - because the first version of it looked like it worked.

Same four calls again, with `ora` where `sql` and `pg` were.

In [ ]:
from connect_ora_instance import connect_ora_instance
from import_ora_table import import_ora_table
from invoke_ora_query import invoke_ora_query

ora_connection = connect_ora_instance(
    instance="127.0.0.1/XEPDB1",
    username="stackexchange",
    password="Passw0rd!"
)

#### First difference: what is *not* in that call

There is no `database=`. For Oracle the service name is part of the instance, so it is `127.0.0.1/XEPDB1` - and `Connect-OraInstance` in the sibling repository has no `-Database` parameter either.

And there was nothing to install. `oracledb` runs in "thin mode": it speaks the Oracle network protocol itself. SQL Server needs the Microsoft ODBC driver installed separately, and the PowerShell version downloads `Oracle.ManagedDataAccess.dll` from nuget.org before it can connect at all. Here `pip install oracledb` is the whole story - no Oracle Instant Client anywhere.

#### And what the columns look like

In [ ]:
cursor = ora_connection.cursor()
cursor.execute('SELECT * FROM "USERS" WHERE 1=0')
ora_description = cursor.description
cursor.close()

for column in ora_description:
    print(f"{column.name:16} {str(column.type_code):28} null_ok={column.null_ok}")

Two things are different here, and neither is about case.

The names are `UPPER CASE`, the exact inverse of PostgreSQL - and the lower casing that both import functions already do for matching handles that without a change.

But the *types* are not Python types. pyodbc reported `int`, `str`, `datetime`: the types it wants to be handed. `oracledb` reports its own, `DB_TYPE_NUMBER` and `DB_TYPE_TIMESTAMP` and `DB_TYPE_VARCHAR`. So the converter table inside `import_ora_table` is keyed on something entirely different from the one inside `import_sql_table`, even though the two functions look identical from the outside.

And `ABOUTME` says `DB_TYPE_LONG`, although the column is a `CLOB`. That is not Oracle being vague - it is already the effect of a decision `connect_ora_instance` made on our behalf, which is the third difference below.

In [ ]:
import_ora_table(
    connection=ora_connection,
    path=r"..\data\stackexchange\Users.xml",
    table="Users",
    batch_size=5000,
    truncate_table=True
)

In [ ]:
invoke_ora_query(
    connection=ora_connection,
    query='SELECT "ID", "DISPLAYNAME", "LOCATION", "CREATIONDATE", "REPUTATION" FROM "USERS" FETCH FIRST 5 ROWS ONLY'
)

#### Second difference: two converters, and a trap underneath them

SQL Server needed a converter for all fourteen columns. PostgreSQL needed none. Oracle needs **two**: it converts the numbers out of their strings on its own, but it will not take an ISO timestamp, because that is not what `NLS_TIMESTAMP_FORMAT` describes. Passing the dates as text fails with `ORA-01843: not a valid month`.

Converting only the two timestamp columns was also the fastest of the variants measured - 0.27 s against 0.45 s for converting everything.

But converting was **not enough**, and this is the part worth remembering.

`oracledb` binds a Python `datetime` as `DB_TYPE_DATE`, and an Oracle `DATE` holds whole seconds. So the fractional seconds were dropped on the way in - no error, no warning, and the right number of rows reported. 12179 of the 12220 rows were wrong. `CreationDate` matched perfectly and hid it completely, because every value in that column happens to end in `.000`.

It was only found by reading the rows back and comparing them against the file, column by column. The fix is one line in `import_ora_table`: `setinputsizes` declaring the `TIMESTAMP` columns.

In [ ]:
# In the file, this user's LastAccessDate is 2011-01-03T17:13:19.040

invoke_ora_query(
    connection=ora_connection,
    query="""
SELECT "ID", TO_CHAR("LASTACCESSDATE", 'YYYY-MM-DD HH24:MI:SS.FF3') AS LastAccessDate
FROM "USERS"
ORDER BY "ID"
FETCH FIRST 5 ROWS ONLY"""
)

In [ ]:
import_ora_table(
    connection=ora_connection,
    path=badges_path,
    table="Badges",
    batch_size=5000,
    truncate_table=True,
    column_map={"CreationDate": "Date"}
)

#### Third difference: a CLOB is a handle that prints like a string

`AboutMe` is a `CLOB`, and by default `oracledb` hands out a `LOB` object rather than a string - a lazy handle that has to be read separately.

The catch is that a `LOB` renders as its own content. So a DataFrame full of them prints exactly as if everything were fine, and then fails the moment the data leaves Python: pyodbc refuses them with `Unknown object type LOB during describe`.

`connect_ora_instance` therefore sets `oracledb.defaults.fetch_lobs = False`, which also measured 17 times faster than fetching handles and reading each one. That default is consulted when a connection is created, which is why it lives in the connect function and not where the rows are read.

In [ ]:
frame = invoke_ora_query(
    connection=ora_connection,
    query='SELECT "ID", "DISPLAYNAME", "ABOUTME" FROM "USERS" WHERE "ABOUTME" IS NOT NULL FETCH FIRST 3 ROWS ONLY'
)

print("type in the cell:", type(frame["ABOUTME"][0]))

frame

### Moving data between databases

Every import so far read a file. Now the source is another table - and it does not have to be in the same database, or even in the same database system.

#### What is a data reader here?

`Get-SqlDataReader` in PowerShell returns a `DbDataReader`: an open result set that hands out one row at a time, so the rows never all sit in memory at once.

In Python a cursor already is exactly that. `get_sql_data_reader` runs the query and returns the cursor, the writer reads it in batches with `fetchmany`, and the writer closes it when it is done - the same ownership as in the sibling, where `Write-SqlTable` disposes the reader it was handed.

In [ ]:
from get_pg_data_reader import get_pg_data_reader
from get_sql_data_reader import get_sql_data_reader
from write_pg_table import write_pg_table
from write_sql_table import write_sql_table

#### Inside one database system

The source gets its own connection, because one connection is reading while the other is writing. The row count is passed in separately: a reader does not know how many rows are still coming, so without it there is no percentage to show.

In [ ]:
source_connection = connect_sql_instance(
    instance="127.0.0.1",
    database="StackExchange",
    username="StackExchange",
    password="Passw0rd!"
)

row_count = invoke_sql_query(
    connection=source_connection,
    query="SELECT COUNT(*) FROM dbo.Users",
    as_type="single_value"
)

data_reader = get_sql_data_reader(connection=source_connection, table="dbo.Users")

write_sql_table(
    connection=connection,
    table="dbo.Import_Users",
    data_reader=data_reader,
    data_reader_row_count=row_count,
    batch_size=2000,
    truncate_table=True
)

# The writer closed the reader, but the connection we opened is ours to close. A reader keeps
# its transaction open, and a connection left in one keeps its locks.
source_connection.close()

#### From PostgreSQL into SQL Server

The same two calls. Only the function that opens the reader is a different one, and the rows now cross from one database system into another.

In [ ]:
row_count = invoke_pg_query(
    connection=pg_connection,
    query="SELECT COUNT(*) FROM Users",
    as_type="single_value"
)

data_reader = get_pg_data_reader(connection=pg_connection, table="Users")

write_sql_table(
    connection=connection,
    table="dbo.Import_Users",
    data_reader=data_reader,
    data_reader_row_count=row_count,
    batch_size=2000,
    truncate_table=True
)

In [ ]:
invoke_sql_query(
    connection=connection,
    query="SELECT TOP 5 Id, DisplayName, Location, CreationDate FROM dbo.Import_Users"
)

Nothing had to be converted on the way. psycopg hands out Python objects, pyodbc takes Python objects, and the dates, numbers and `NULL`s arrive as what they were.

#### And back the other way

Same shape again, and this one is the fastest of the four, because the target is PostgreSQL and `write_pg_table` feeds the rows straight into `COPY`.

In [ ]:
row_count = invoke_sql_query(
    connection=connection,
    query="SELECT COUNT(*) FROM dbo.Users",
    as_type="single_value"
)

data_reader = get_sql_data_reader(connection=connection, table="dbo.Users")

write_pg_table(
    connection=pg_connection,
    table="Import_Users",
    data_reader=data_reader,
    data_reader_row_count=row_count,
    batch_size=2000,
    truncate_table=True
)

In [ ]:
invoke_pg_query(
    connection=pg_connection,
    query="SELECT id, displayname, location, creationdate FROM Import_Users LIMIT 5"
)

#### Now the third system, in both directions

With three databases there are nine directions, and the call sites are all the same two calls. Two of them are worth showing, because they are the ones that needed something the other seven did not.

First into Oracle. `write_ora_table` looks like `write_sql_table` rather than `write_pg_table`, because Oracle has no `COPY` - so it is `executemany` in batches again.

In [ ]:
from get_ora_data_reader import get_ora_data_reader
from write_ora_table import write_ora_table

row_count = invoke_pg_query(
    connection=pg_connection,
    query="SELECT COUNT(*) FROM Users",
    as_type="single_value"
)

data_reader = get_pg_data_reader(connection=pg_connection, table="Users")

write_ora_table(
    connection=ora_connection,
    table="Import_Users",
    data_reader=data_reader,
    data_reader_row_count=row_count,
    batch_size=2000,
    truncate_table=True
)

In [ ]:
# The milliseconds crossed from PostgreSQL into Oracle intact - because write_ora_table
# declares the TIMESTAMP columns too

invoke_ora_query(
    connection=ora_connection,
    query="""
SELECT "ID", TO_CHAR("LASTACCESSDATE", 'YYYY-MM-DD HH24:MI:SS.FF3') AS LastAccessDate
FROM "IMPORT_USERS"
ORDER BY "ID"
FETCH FIRST 5 ROWS ONLY"""
)

#### And out of Oracle again

This is the direction that carries the `CLOB`. It is also the one that failed until `connect_ora_instance` started asking for strings instead of `LOB` handles - `write_pg_table` and `write_sql_table` would both have been handed an object neither driver can describe.

In [ ]:
ora_source_connection = connect_ora_instance(
    instance="127.0.0.1/XEPDB1",
    username="stackexchange",
    password="Passw0rd!"
)

row_count = invoke_ora_query(
    connection=ora_source_connection,
    query='SELECT COUNT(*) FROM "USERS"',
    as_type="single_value"
)

data_reader = get_ora_data_reader(connection=ora_source_connection, table="Users")

write_pg_table(
    connection=pg_connection,
    table="Import_Users",
    data_reader=data_reader,
    data_reader_row_count=row_count,
    batch_size=2000,
    truncate_table=True
)

ora_source_connection.close()

In [ ]:
invoke_pg_query(
    connection=pg_connection,
    query="SELECT id, displayname, lastaccessdate FROM Import_Users ORDER BY id LIMIT 5"
)

All nine directions of the same 12220 rows, measured on this machine. The first four were measured together and the Oracle five in a later run, so compare within a block rather than across them:

| | |
| --- | --- |
| SQL Server to SQL Server | 1.00 s |
| PostgreSQL to SQL Server | 0.88 s |
| SQL Server to PostgreSQL | 0.31 s |
| PostgreSQL to PostgreSQL | 0.16 s |
| | |
| Oracle to Oracle | 0.89 s |
| SQL Server to Oracle | 0.61 s |
| PostgreSQL to Oracle | 0.48 s |
| Oracle to SQL Server | 1.03 s |
| Oracle to PostgreSQL | 0.50 s |

The ones that write into PostgreSQL are the quick ones, for the same reason the file import was: `COPY`. And in none of them was the whole table ever in memory - the writer asked the reader for the next batch, wrote it, and asked again.

Nothing had to be converted between the systems either. Each driver hands out Python objects and each driver takes Python objects, so the dates, numbers and `NULL`s cross unchanged.

With two exceptions, and both of them are Oracle. Writing into it needs the `TIMESTAMP` columns declared, or the milliseconds vanish without a word. Reading out of it needs `fetch_lobs = False`, or the `CLOB` arrives as a handle that prints like a string and then cannot be written anywhere. Two lines, in two functions - and neither problem announces itself.

### Bonus: into a database that has no schema

Every target so far had columns, and every import function starts the same way: ask the target what its columns are and what type each one wants, then make the values fit.

MongoDB has nothing to ask. That changes where the work happens, and it is the reason this section is worth the last five minutes.

In [ ]:
from connect_mdb_instance import connect_mdb_instance
from read_mdb_collection import read_mdb_collection
from write_mdb_collection import write_mdb_collection

mdb_connection = connect_mdb_instance(
    instance="127.0.0.1",
    database="stackexchange",
    username="stackexchange",
    password="Passw0rd!"
)

mdb_connection

What came back is not a connection - it is a *database*. `pymongo` hands out a client, a database and a collection as three separate objects, and the database is the one everything else hangs off: `mdb_connection["Users"]` is the collection. The sibling returns all three in a `PSCustomObject`, because the Mdbc module needs all three.

Two consequences, and the second one is the interesting kind.

A database has no `close()`. The client behind it does, which is the last cell of this notebook.

And `MongoClient` does not talk to the server until the first operation. Without the `ping` that `connect_mdb_instance` does, the cell above would have succeeded with no MongoDB running at all - and `06_test_connections.py` would have reported a healthy connection to nothing.

#### The types have nowhere to come from

For SQL Server the converter came from `cursor.description`. For Oracle, from the two `TIMESTAMP` columns it reported. For PostgreSQL there was no converter at all, because `COPY` let the server parse the text.

Here there is nothing to ask. So the conversion moves out of the function and into the open - which is exactly what the PowerShell version does as well, and it is why `write_mdb_collection` is the shortest of the five write functions. It inserts what it is handed and knows nothing about types.

In [ ]:
import datetime

users_documents = [
    {
        "_id": int(row["Id"]),
        "Reputation": int(row["Reputation"]),
        "CreationDate": datetime.datetime.fromisoformat(row["CreationDate"]),
        "DisplayName": row.get("DisplayName"),
        "LastAccessDate": datetime.datetime.fromisoformat(row["LastAccessDate"]),
        "WebsiteUrl": row.get("WebsiteUrl"),
        "Location": row.get("Location"),
        "AboutMe": row.get("AboutMe"),
        "Views": int(row["Views"]),
        "UpVotes": int(row["UpVotes"]),
        "DownVotes": int(row["DownVotes"]),
        "AccountId": int(row["AccountId"])
    }
    for row in (
        ET.fromstring(line).attrib
        for line in users_lines
        if line.lstrip().startswith("<row")
    )
]

users_documents[0]

`Id` became `_id`, which is MongoDB's own primary key - the same rename the sibling does.

And the optional attributes are asked for with `.get()`, so a user without a `Location` gets `None`. That is the very same `.get()` the first section of this notebook needed, for the very same reason: a missing attribute is simply not in the dictionary.

In [ ]:
write_mdb_collection(
    connection=mdb_connection,
    collection="Users",
    data=users_documents,
    batch_size=5000,
    truncate_collection=True
)

#### Reading it back

The filter, the projection and the sort are plain dictionaries here, where the sibling passes hashtables. So `-Filter @{ Location = 'Canada' }` becomes `filter={"Location": "Canada"}`, and the call site stays recognisable across the two languages.

This is the sibling's own demo query, line for line.

In [ ]:
read_mdb_collection(
    connection=mdb_connection,
    collection="Users",
    filter={"Location": "Canada"},
    first=5,
    project={"CreationDate": 1, "DisplayName": 1, "Location": 1}
)

In [ ]:
read_mdb_collection(
    connection=mdb_connection,
    collection="Users",
    project={"DisplayName": 1, "Reputation": 1},
    sort=[("Reputation", -1)],
    first=5
)

The dates came back as dates and the numbers as numbers, because that is how they went in. Nothing in `write_mdb_collection` or `read_mdb_collection` knows anything about a type.

MongoDB stores a date with millisecond precision, and these files have exactly three fractional digits, so all 12220 of them survive the round trip unchanged. Worth checking rather than assuming - the Oracle section above is what happens when nobody does.

In [ ]:
# A database has nothing to close - the client behind it does

mdb_connection.client.close()